In [ ]:
"""
Preprocessing for survivor calls.
"""

import importlib


import pysam
import os


workdir = "/vast/ai_projects/Papenfuss_SVEnsemble2"
os.chdir(workdir)
os.getcwd()

import svensemble.vcfparse as vcfp # noqa: E402 (flake8)
import svensemble.breakends as bnds
import svensemble.breakpoints as bps
import svensemble.events as evts
import svensemble.caller_specific as caller_specific
import svensemble.benchmark as benchmark
import svensemble._caller_params as caller_params
import svensemble._sample_params as sample_params
import svensemble.chunk_match as chunk_match
import svensemble.matched_breakpoints as matchedbps
import svensemble.utils as sv_utils

importlib.reload(vcfp)
importlib.reload(bnds)
importlib.reload(bps)
importlib.reload(evts)
importlib.reload(caller_specific)
importlib.reload(benchmark)
importlib.reload(caller_params)
importlib.reload(sample_params)
importlib.reload(chunk_match)
importlib.reload(matchedbps)
importlib.reload(sv_utils)

sample_names = ['NA12878_30x', 'NA12878', 'CHM13', 'HG002']
model_names = ['unbiased', 'unbiased_GT']
for model_name in model_names:
    for sample_name in sample_names:
        sample_config = sv_utils.create_config_dict(sample_name, model_name=model_name, extra_callsets=['survivor'])
        survivor_path = sample_config[sample_name]['callsets']['survivor']['path']
        survivor_params = sample_config[sample_name]['callsets']['survivor']['params']
        survivor_vcf_df, _ = vcfp.parse_vcf_to_df(pysam.VariantFile(survivor_path), caller_params=survivor_params)
        survivor_vcf_df.loc[:, ['INFO_CIPOS_l', 'INFO_CIPOS_r']] = sv_utils.split_cipos(survivor_vcf_df['INFO_CIPOS'])
        survivor_vcf_df.loc[:, ['INFO_CIEND_l', 'INFO_CIEND_r']] = sv_utils.split_ciend(survivor_vcf_df['INFO_CIEND'])
        survivor_vcf_df = survivor_vcf_df.drop(columns=['INFO_CIPOS', 'INFO_CIEND'])
        survivor_vcf_df.loc[:, 'INFO_EVENT'] = survivor_vcf_df['ID']
        survivor_bnds = evts.event_to_bnd(survivor_vcf_df, bnd2_pos=survivor_params['bnd2_pos'])
        survivor_bps = bnds.bnd_to_bp(survivor_bnds)
        bps_path = sample_config[sample_name]['callsets']['survivor']['svensemble_dir'] / f"survivor.{model_name}.bps.csv"
        if not os.path.exists(bps_path.parent):
            os.makedirs(bps_path.parent)
        survivor_bps.to_csv(bps_path)